In [7]:
import os, json, time, random
from io import StringIO
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import netCDF4 as nc
import earthaccess


# -----------------------------
# SETUP
# -----------------------------
GPKG_PATH = r"./dnipro_sword_reaches_clip.gpkg"
LAYER_NAME = "dnipro_reaches"
REACH_ID_FIELD = "reach_id"

# IMPORTANT: your TEST folder
OUT_DIR = r"./riverSP_DAWG_reach_TEST"

HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_2.0"
FIELDS = "time_str,cycle_id,pass_id,wse,slope,width"

TIMEOUT_S = 60
SLEEP_BETWEEN_REQUESTS_S = (0.2, 0.6)
MAX_RETRIES = 5

OVERLAP_HOURS = 48
DEFAULT_BACKFILL_DAYS_IF_EMPTY = 2

# DAWG / Earthaccess
SHORT_NAME = "SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3"
DOWNLOAD_DIR = "./downloaded_files"
DAWG_REGION_PREFIX = "eu_"


# -----------------------------
# Helpers
# -----------------------------
def iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def safe_reach_filename(reach_id: str) -> str:
    s = str(reach_id).strip()
    cleaned = "".join(ch for ch in s if ch.isalnum() or ch in ("-", "_"))
    return cleaned or "unknown_reach"


def load_reach_ids() -> list[str]:
    gdf = gpd.read_file(GPKG_PATH, layer=LAYER_NAME)
    if REACH_ID_FIELD not in gdf.columns:
        raise ValueError(f"Field '{REACH_ID_FIELD}' not found in layer '{LAYER_NAME}'")
    s = gdf[REACH_ID_FIELD].dropna().astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    return s.loc[s != ""].unique().tolist()


def read_existing_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def get_last_time_dt_from_time_utc(existing: pd.DataFrame) -> datetime | None:
    """
    Reads the last timestamp from time_utc (ISO 'Z' strings), returns UTC datetime.
    """
    if existing is None or len(existing) == 0 or "time_utc" not in existing.columns:
        return None

    s = existing["time_utc"].astype(str).str.strip()
    s = s[(s != "") & (s.str.lower() != "nan")]
    if len(s) == 0:
        return None

    ts = pd.to_datetime(s, errors="coerce", utc=True)
    if ts.notna().any():
        return ts.max().to_pydatetime()
    return None


# -----------------------------
# Hydrocron
# -----------------------------
def hydrocron_response_to_df(text: str) -> pd.DataFrame:
    text = (text or "").strip()
    if not text:
        return pd.DataFrame()
    if text.startswith("{"):
        try:
            obj = json.loads(text)
        except json.JSONDecodeError:
            return pd.DataFrame()
        csv_text = (obj.get("results", {}).get("csv", "") or "").strip()
        return pd.read_csv(StringIO(csv_text)) if csv_text else pd.DataFrame()
    return pd.read_csv(StringIO(text))


def request_with_retries(url: str, params: dict) -> requests.Response:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, params=params, timeout=TIMEOUT_S)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}")
            return r
        except Exception as e:
            last_err = e
            backoff = min(30, (2 ** (attempt - 1)) * 0.7) + random.uniform(0, 0.5)
            time.sleep(backoff)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err}")


def normalize_hydrocron_new(df: pd.DataFrame, rid: str) -> pd.DataFrame:
    """
    Convert Hydrocron response to internal form:
      _dt (UTC datetime), plus hydro vars
    """
    cols = ["_dt", "cycle_id", "pass_id", "wse", "slope", "width"]
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=cols)

    df = df.copy()
    if "time_str" not in df.columns:
        return pd.DataFrame(columns=cols)

    t = pd.to_datetime(df["time_str"], errors="coerce", utc=True)
    df = df.loc[t.notna()].copy()
    df["_dt"] = t.loc[t.notna()].values

    keep = [c for c in cols if c in df.columns]
    return df[keep].copy()


def fetch_hydrocron_window(rid: str, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    params = {
        "feature": "Reach",
        "feature_id": rid,
        "start_time": iso_z(start_dt),
        "end_time": iso_z(end_dt),
        "output": "csv",
        "collection_name": COLLECTION_NAME,
        "fields": FIELDS,
    }
    time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS_S))
    r = request_with_retries(HYDROCRON_URL, params)
    raw = hydrocron_response_to_df(r.text)
    return normalize_hydrocron_new(raw, rid)


# -----------------------------
# DAWG consensus_q (Earthaccess NetCDF)
# -----------------------------
def download_latest_dawg_eu_netcdf() -> str:
    earthaccess.login(strategy="netrc")
    granules = earthaccess.search_data(short_name=SHORT_NAME, count=500)
    eu_granules = [g for g in granules if g["meta"]["native-id"].startswith(DAWG_REGION_PREFIX)]
    if not eu_granules:
        raise RuntimeError(f"No DAWG SOS granules found with native-id starting '{DAWG_REGION_PREFIX}'.")

    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    paths = earthaccess.download(eu_granules, local_path=DOWNLOAD_DIR)
    if not paths:
        raise RuntimeError("earthaccess.download returned no files.")

    paths_sorted = sorted(paths, key=lambda p: os.path.basename(p))
    return paths_sorted[-1]


def open_dawg(nc_path: str):
    ds = nc.Dataset(nc_path, "r")
    reaches = ds.groups["reaches"]
    consensus = ds.groups["consensus"]

    nc_reach_ids = reaches.variables["reach_id"][:].astype("int64")
    id_to_idx = {int(rid): int(i) for i, rid in enumerate(nc_reach_ids)}

    qvar = consensus.variables["consensus_q"]
    time_var = consensus.variables["time_int"]

    missing = None
    if "_FillValue" in qvar.ncattrs():
        missing = qvar.getncattr("_FillValue")
    elif "missing_value" in qvar.ncattrs():
        missing = qvar.getncattr("missing_value")

    return ds, id_to_idx, time_var, qvar, missing


def fetch_dawg_window(
    rid: str, id_to_idx: dict, time_var, qvar, missing, start_dt: datetime, end_dt: datetime
) -> pd.DataFrame:
    cols = ["_dt", "consensus_q"]
    try:
        rid_int = int(float(str(rid).strip()))
    except Exception:
        return pd.DataFrame(columns=cols)

    if rid_int not in id_to_idx:
        return pd.DataFrame(columns=cols)

    i = id_to_idx[rid_int]

    times = np.asarray(time_var[i], dtype="float64")
    valid_time = times > -9.0e10
    times_valid = times[valid_time].astype("int64")
    if times_valid.size == 0:
        return pd.DataFrame(columns=cols)

    dt64 = np.array([np.datetime64("2000-01-01T00:00:00") + np.timedelta64(int(t), "s") for t in times_valid])
    dt = pd.to_datetime(dt64.astype("datetime64[ns]"), utc=True)

    q = np.asarray(qvar[i], dtype="float64")[valid_time]
    if missing is not None:
        q[q == missing] = np.nan
    q[q <= -9.0e10] = np.nan

    df = pd.DataFrame({"_dt": dt, "consensus_q": q}).dropna(subset=["_dt"])

    start_dt = start_dt.astimezone(timezone.utc)
    end_dt = end_dt.astimezone(timezone.utc)
    return df[(df["_dt"] >= start_dt) & (df["_dt"] <= end_dt)].copy()


# -----------------------------
# Append / dedupe using time_utc
# -----------------------------
def append_dedup_time_utc(existing: pd.DataFrame, new_chunk: pd.DataFrame) -> pd.DataFrame:
    if existing is None or len(existing) == 0:
        out = new_chunk.copy()
    elif new_chunk is None or len(new_chunk) == 0:
        out = existing.copy()
    else:
        out = pd.concat([existing, new_chunk], ignore_index=True, sort=False)

    if "time_utc" in out.columns:
        out["time_utc"] = out["time_utc"].astype(str)
        out = out.drop_duplicates(subset=["time_utc"], keep="last")
        out = out.sort_values("time_utc")

    return out


def build_output_chunk(rid: str, hydro_df: pd.DataFrame, dawg_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge hydro + dawg on _dt, then write to output format with time_utc only.
    """
    hydro_df = hydro_df.copy()
    dawg_df = dawg_df.copy()

    hydro_df["_dt"] = pd.to_datetime(hydro_df.get("_dt"), errors="coerce", utc=True)
    dawg_df["_dt"] = pd.to_datetime(dawg_df.get("_dt"), errors="coerce", utc=True)

    if len(hydro_df) == 0 and len(dawg_df) == 0:
        return pd.DataFrame(columns=["reach_id","time_utc","cycle_id","pass_id","wse","slope","width","consensus_q"])

    if len(hydro_df) == 0:
        merged = dawg_df.copy()
    elif len(dawg_df) == 0:
        merged = hydro_df.copy()
    else:
        merged = pd.merge(hydro_df, dawg_df[["_dt","consensus_q"]], on="_dt", how="outer")

    merged["reach_id"] = str(rid)
    merged = merged.drop_duplicates(subset=["_dt"], keep="last").sort_values("_dt")

    merged["time_utc"] = pd.to_datetime(merged["_dt"], utc=True, errors="coerce").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    merged = merged.drop(columns=["_dt"], errors="ignore")

    # Nice column order (exactly like historical)
    desired = ["reach_id","time_utc","cycle_id","pass_id","wse","slope","width","consensus_q"]
    for c in desired:
        if c not in merged.columns:
            merged[c] = pd.NA
    return merged[desired]


def main():
    end_dt = datetime.now(timezone.utc)

    reach_ids = load_reach_ids()
    os.makedirs(OUT_DIR, exist_ok=True)

    print("Append run (time_utc-based)")
    print("Reaches:", len(reach_ids))
    print("OUT_DIR:", OUT_DIR)
    print("End time:", iso_z(end_dt))

    # Open DAWG once
    try:
        nc_path = download_latest_dawg_eu_netcdf()
        print("Using DAWG NetCDF:", nc_path)
        ds, id_to_idx, time_var, qvar, missing = open_dawg(nc_path)
    except Exception as e:
        ds = None
        id_to_idx = time_var = qvar = missing = None
        print("WARNING: DAWG failed; will append Hydrocron only.")
        print("  Error:", e)

    appended = 0
    no_change = 0
    errors = 0

    try:
        for i, rid in enumerate(reach_ids, 1):
            out_csv = os.path.join(OUT_DIR, f"reach_{safe_reach_filename(rid)}.csv")
            existing = read_existing_csv(out_csv)

            last_dt = get_last_time_dt_from_time_utc(existing)

            if last_dt is not None:
                start_dt = last_dt - timedelta(hours=OVERLAP_HOURS)
            else:
                start_dt = end_dt - timedelta(days=DEFAULT_BACKFILL_DAYS_IF_EMPTY)

            if start_dt >= end_dt:
                start_dt = end_dt - timedelta(days=1)

            try:
                hydro_new = fetch_hydrocron_window(rid, start_dt, end_dt)

                if ds is not None:
                    dawg_new = fetch_dawg_window(rid, id_to_idx, time_var, qvar, missing, start_dt, end_dt)
                else:
                    dawg_new = pd.DataFrame(columns=["_dt","consensus_q"])

                new_chunk = build_output_chunk(rid, hydro_new, dawg_new)

                # If nothing returned at all
                if len(new_chunk) == 0:
                    no_change += 1
                    print(f"[{i}/{len(reach_ids)}] {rid} no_change (hydro=0 dawg=0)")
                    continue

                # Determine if truly new vs existing
                if existing is not None and len(existing) > 0 and "time_utc" in existing.columns:
                    existing_times = set(existing["time_utc"].astype(str).tolist())
                    incoming_times = set(new_chunk["time_utc"].astype(str).tolist())
                    if len(incoming_times - existing_times) == 0:
                        no_change += 1
                        print(f"[{i}/{len(reach_ids)}] {rid} no_change (hydro={len(hydro_new)} dawg={len(dawg_new)}; all timestamps already present)")
                        continue

                out_df = append_dedup_time_utc(existing, new_chunk)
                out_df.to_csv(out_csv, index=False)

                appended += 1
                print(f"[{i}/{len(reach_ids)}] {rid} appended hydro={len(hydro_new)} dawg={len(dawg_new)} total={len(out_df)}")

            except Exception as e:
                errors += 1
                print(f"[{i}/{len(reach_ids)}] {rid} ERROR: {e}")

    finally:
        if ds is not None:
            ds.close()

    print("\nSUMMARY")
    print(f"  appended : {appended}")
    print(f"  no change: {no_change}")
    print(f"  errors   : {errors}")
    print("Done.")


if __name__ == "__main__":
    main()

Append run (time_utc-based)
Reaches: 842
OUT_DIR: ./riverSP_DAWG_reach_TEST
End time: 2026-03-05T20:32:20Z


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Using DAWG NetCDF: downloaded_files/eu_sword_v16_SOS_results_unconstrained_20230502T204408_20250502T204408_20251219T163700.nc
[1/842] 22511300103 appended hydro=24 dawg=0 total=169
[2/842] 22400900035 appended hydro=34 dawg=0 total=187
[3/842] 22601000045 no_change (hydro=0 dawg=0)
[4/842] 22511100015 no_change (hydro=0 dawg=0)
[5/842] 22511100055 appended hydro=25 dawg=0 total=154
[6/842] 22511100021 appended hydro=23 dawg=0 total=157
[7/842] 22511100031 appended hydro=25 dawg=0 total=166
[8/842] 22511100041 no_change (hydro=0 dawg=0)
[9/842] 22511300011 appended hydro=25 dawg=0 total=166
[10/842] 22511300291 no_change (hydro=0 dawg=0)
[11/842] 22511300021 appended hydro=25 dawg=25 total=167
[12/842] 22511300281 no_change (hydro=0 dawg=0)
[13/842] 22511200011 appended hydro=25 dawg=16 total=166
[14/842] 22511300301 no_change (hydro=0 dawg=0)
[15/842] 22511300031 appended hydro=33 dawg=23 total=197
[16/842] 22511200021 appended hydro=25 dawg=15 total=165
[17/842] 22511300063 no_change 

/tmp/ipykernel_3822844/1247216823.py:240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([existing, new_chunk], ignore_index=True, sort=False)


[466/842] 22520400221 appended hydro=17 dawg=0 total=140
[467/842] 22520400171 appended hydro=32 dawg=0 total=196
[468/842] 22520400091 appended hydro=32 dawg=0 total=197


/tmp/ipykernel_3822844/1247216823.py:240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([existing, new_chunk], ignore_index=True, sort=False)


[469/842] 22520400211 appended hydro=17 dawg=0 total=132
[470/842] 22520400191 appended hydro=17 dawg=0 total=142
[471/842] 22541000011 appended hydro=32 dawg=0 total=111
[472/842] 22542000031 appended hydro=32 dawg=0 total=114
[473/842] 22542000021 appended hydro=32 dawg=0 total=113
[474/842] 22542000041 appended hydro=32 dawg=0 total=114
[475/842] 22541000021 appended hydro=32 dawg=0 total=113
[476/842] 22542000056 no_change (hydro=0 dawg=0)
[477/842] 22542000011 appended hydro=32 dawg=0 total=113
[478/842] 22543000011 appended hydro=32 dawg=0 total=112
[479/842] 22543000021 appended hydro=33 dawg=0 total=112
[480/842] 22543000031 appended hydro=33 dawg=0 total=113
[481/842] 22520100141 appended hydro=24 dawg=8 total=82
[482/842] 22550000021 appended hydro=16 dawg=5 total=54
[483/842] 22520100151 appended hydro=32 dawg=14 total=111
[484/842] 22550000031 appended hydro=17 dawg=0 total=53
[485/842] 22520300041 appended hydro=17 dawg=0 total=40
[486/842] 22520100161 appended hydro=33 da

KeyboardInterrupt: 